# 14 — GridSearchCV tuning, docs-driven grids (builds on notebooks 09-12)

**Why this notebook exists, separate from notebook 12:** the coach reviewed the `RandomizedSearchCV`
tuning code in notebook 12 directly and asked for two changes: (1) always check the actual
scikit-learn/XGBoost documentation for what's worth tuning rather than picking a handful of values
ad hoc, and (2) use `GridSearchCV` (exhaustive) instead of `RandomizedSearchCV` (sampled) so the
whole grid actually gets searched, not just a random sample of it.

**This notebook does not touch or replace notebook 12.** Same data, same selected features from
notebook 11 (`selected_feature_cols_all_vehicles.json`, the 53 rolling-mean/lag/calendar features
-- this is *not* related to the individual-lag experiment in notebook 13, which already came back
worse and was shelved), same F2-scored evaluation on the same held-out test split. The only change
is the search itself: full grid, more folds, more values per parameter. Everything saves under new
filenames so notebook 12's saved models/results are untouched.

**Runtime warning, read before running:** this grid is large by design (not trimmed, per instruction).
RF: 5 x 4 x 3 x 2 = 120 param combinations x 5 folds = 600 model fits, some with up to 500 trees on
~140,000 rows. XGBoost: the same 4-dimension shape, another 600 fits. **This will realistically take
hours, likely run overnight** -- not something to run inside a quick interactive session. See the
"How to run this" note at the end of this cell.

**How to run this (on your own machine, not in a hosted sandbox):**
1. `pip install -r requirements.txt` if you haven't already (need scikit-learn, xgboost, pandas).
2. From the repo root: `jupyter nbconvert --to notebook --execute --inplace notebooks/14_gridsearch_tuning_all_vehicles.ipynb`
   -- run this in a terminal you can leave open overnight (or `nohup ... &` if you want to close the
   terminal and check back later; redirect output to a log file so you can watch progress:
   `nohup jupyter nbconvert --to notebook --execute --inplace notebooks/14_gridsearch_tuning_all_vehicles.ipynb > nb14_run.log 2>&1 &`).
3. Make sure your machine doesn't sleep overnight (disable sleep/screen-lock-triggered suspend --
   a laptop going to sleep will pause or kill the job).
4. Both `GridSearchCV` calls use `verbose=2`, so `nb14_run.log` will show progress (fold-by-fold
   fit timing) while it runs, not just silence until the end.
5. When it's done, the notebook file itself will have all outputs saved in place -- send it back /
   let me know and I'll read the results straight out of it.

**One honest heads-up before you spend the compute:** both grids include a `scaler` on/off toggle
(`StandardScaler()` vs `None`) per the coach's snippet. Worth knowing going in -- tree-based models
(Random Forest, XGBoost) split on raw feature thresholds, and `StandardScaler` is a monotonic
per-feature rescaling, so it mathematically cannot change which splits a tree picks or what it
predicts. This dimension is expected to come back a coin-flip / no real difference between the two
options for both models -- worth mentioning to the coach as a real thing we checked and understood,
not just cargo-culted from a template that assumes a scale-sensitive model like logistic regression
or SVM. Kept in the grid as asked, not removed, since it's genuinely harmless to include (just some
extra compute) and it's a good thing to be able to show and explain if asked.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, fbeta_score,
                              f1_score, recall_score, precision_score, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=True)

## 1. Load — selected features from notebook 11, same stratified split as notebook 12

In [2]:
df = pd.read_csv("../data/processed/all_vehicles_features.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

with open("../data/processed/selected_feature_cols_all_vehicles.json") as f:
    SELECTED_FEATURE_COLS = json.load(f)

TARGET = "Fault_Within_6h"
df[TARGET] = df[TARGET].astype(bool)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=42)
X_train, y_train = train_df[SELECTED_FEATURE_COLS], train_df[TARGET]
X_test, y_test = test_df[SELECTED_FEATURE_COLS], test_df[TARGET]

print(f"Train: {len(X_train):,} rows, Test: {len(X_test):,} rows, {len(SELECTED_FEATURE_COLS)} features")

Train: 140,067 rows, Test: 35,017 rows, 53 features


## 2. Random Forest — full GridSearchCV, docs-informed grid (coach's spec, bugs fixed)

Fixes applied to the pasted version: `"scalar"` -> `"scaler"` (the pipeline step is named
`"scaler"`; the typo would make `GridSearchCV` crash trying to set a param that doesn't exist),
`random_state` added back to both `StratifiedKFold` and `RandomForestClassifier` for reproducibility,
and the commented-out manual refit block removed -- `GridSearchCV` already refits the winning
combination on the full training set by default (`refit=True` is the default), so
`rf_search.best_estimator_` is already a fitted model, nothing extra to do.

5-fold (not 10) chosen as a reasonable middle ground: 10-fold would double every fit count below for
a study that's already large by design.

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_param_space = {
    "clf__n_estimators": [100, 200, 300, 400, 500],
    "clf__max_depth": [5, 6, 7, 8],
    "clf__min_samples_leaf": [2, 4, 6],
    "scaler": [StandardScaler(), None],  # kept per instruction -- see runtime note above
}

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
])

rf_search = GridSearchCV(
    rf_pipeline,
    rf_param_space,
    cv=skf,
    scoring=f2_scorer,
    n_jobs=-1,
    verbose=2,
)
rf_search.fit(X_train, y_train)

print(f"Best RF params: {rf_search.best_params_}")
print(f"Best RF CV F2 (full training set): {rf_search.best_score_:.6f}")

rf_best = rf_search.best_estimator_  # GridSearchCV refit=True by default -- already trained on the full training set
print("GridSearchCV already refit rf_best on the full training set (refit=True is the default).")

Fitting 5 folds for each of 120 candidates, totalling 600 fits


Best RF params: {'clf__max_depth': 8, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 100, 'scaler': None}
Best RF CV F2 (full training set): 0.481065
GridSearchCV already refit rf_best on the full training set (refit=True is the default).


## 3. XGBoost — comparable full GridSearchCV, grid pulled from XGBoost's own docs

XGBoost's own "Notes on Parameter Tuning" documentation names `n_estimators` (`num_boost_round`),
`max_depth`, and `learning_rate` (`eta`) as the primary levers for the bias/variance and
speed/accuracy trade-off -- the boosting equivalents of RF's tree-count/depth/leaf-size dimensions.
Kept the grid the same size and shape as RF's (4 dimensions, 120 combinations, same 5-fold CV,
same `scaler` toggle for the same reason) for a fair side-by-side, rather than copying RF's grid
values directly, since RF and XGBoost don't share the same hyperparameters
(RF has no learning rate; XGBoost's `min_samples_leaf` equivalent is `min_child_weight`, not used
here to keep the comparison the same size -- a natural next grid to try if this one doesn't move
the needle enough).

In [4]:
neg, pos = (~y_train).sum(), y_train.sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {neg}/{pos} = {scale_pos_weight:.6f}")

xgb_param_space = {
    "clf__n_estimators": [100, 200, 300, 400, 500],
    "clf__max_depth": [3, 4, 5, 6],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "scaler": [StandardScaler(), None],  # kept per instruction -- see runtime note above
}

xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=scale_pos_weight, random_state=42,
    )),
])

xgb_search = GridSearchCV(
    xgb_pipeline,
    xgb_param_space,
    cv=skf,
    scoring=f2_scorer,
    n_jobs=-1,
    verbose=2,
)
xgb_search.fit(X_train, y_train)

print(f"Best XGB params: {xgb_search.best_params_}")
print(f"Best XGB CV F2 (full training set): {xgb_search.best_score_:.6f}")

xgb_best = xgb_search.best_estimator_  # already refit on the full training set
print("GridSearchCV already refit xgb_best on the full training set (refit=True is the default).")

scale_pos_weight = 127383/12684 = 10.042810
Fitting 5 folds for each of 120 candidates, totalling 600 fits


Best XGB params: {'clf__learning_rate': 0.1, 'clf__max_depth': 6, 'clf__n_estimators': 500, 'scaler': StandardScaler()}
Best XGB CV F2 (full training set): 0.668771
GridSearchCV already refit xgb_best on the full training set (refit=True is the default).


## 4. Evaluate on the held-out test set

In [5]:
rf_pred = rf_best.predict(X_test)
xgb_pred = xgb_best.predict(X_test)

for name, pred in [("Random Forest (GridSearchCV)", rf_pred), ("XGBoost (GridSearchCV)", xgb_pred)]:
    print("=" * 55)
    print(name.upper())
    print("=" * 55)
    print(classification_report(y_test, pred, labels=[False, True], digits=6, zero_division=0))

RANDOM FOREST (GRIDSEARCHCV)


              precision    recall  f1-score   support

       False   0.966576  0.688313  0.804050     31846
        True   0.195559  0.760959  0.311154      3171

    accuracy                       0.694891     35017
   macro avg   0.581067  0.724636  0.557602     35017
weighted avg   0.896755  0.694891  0.759415     35017

XGBOOST (GRIDSEARCHCV)
              precision    recall  f1-score   support

       False   0.983830  0.880739  0.929434     31846
        True   0.416411  0.854620  0.559975      3171

    accuracy                       0.878373     35017
   macro avg   0.700120  0.867679  0.744705     35017
weighted avg   0.932446  0.878373  0.895977     35017



## 5. Final scoreboard — vs. notebook 12's RandomizedSearchCV results

Quoting notebook 12's already-saved numbers for an honest side-by-side, not recomputing them.

In [6]:
with open("../models/baseline_results_all_vehicles.json") as f:
    randomized_results = json.load(f)

gridsearch_results = {}
for name, pred in [("Random Forest (GridSearchCV, full grid)", rf_pred),
                    ("XGBoost (GridSearchCV, full grid)", xgb_pred)]:
    gridsearch_results[name] = {
        "recall_fault": recall_score(y_test, pred, pos_label=True, zero_division=0),
        "precision_fault": precision_score(y_test, pred, pos_label=True, zero_division=0),
        "F2_fault": fbeta_score(y_test, pred, beta=2, pos_label=True, zero_division=0),
        "F1_fault_secondary": f1_score(y_test, pred, pos_label=True, zero_division=0),
    }

combined = {**randomized_results, **gridsearch_results}
scoreboard = pd.DataFrame(combined).T.round(6)

with open("../models/gridsearch_results_all_vehicles.json", "w") as f:
    json.dump(gridsearch_results, f, indent=2)

scoreboard

,recall_fault,precision_fault,F2_fault,F1_fault_secondary
"Logistic Regression (baseline, notebook 09)",0.516556,0.132987,0.327587,0.211519
"Random Forest (tuned, selected features)",0.749606,0.195799,0.478771,0.310496
"XGBoost (tuned, selected features)",0.853989,0.268013,0.594172,0.407985
"Random Forest (GridSearchCV, full grid)",0.760959,0.195559,0.482156,0.311154
"XGBoost (GridSearchCV, full grid)",0.854620,0.416411,0.706023,0.559975


In [7]:
rf_delta = gridsearch_results["Random Forest (GridSearchCV, full grid)"]["F2_fault"] - \
           randomized_results["Random Forest (tuned, selected features)"]["F2_fault"]
xgb_delta = gridsearch_results["XGBoost (GridSearchCV, full grid)"]["F2_fault"] - \
            randomized_results["XGBoost (tuned, selected features)"]["F2_fault"]

print(f"RF  F2 delta (GridSearchCV vs. notebook 12's RandomizedSearchCV): {rf_delta:+.6f}")
print(f"XGB F2 delta (GridSearchCV vs. notebook 12's RandomizedSearchCV): {xgb_delta:+.6f}")
print()
print("Which scaler setting won each search (expected to be close to a coin flip for both --")
print("tree splits don't depend on feature scale):")
print(f"  RF best scaler:  {rf_search.best_params_['scaler']}")
print(f"  XGB best scaler: {xgb_search.best_params_['scaler']}")

RF  F2 delta (GridSearchCV vs. notebook 12's RandomizedSearchCV): +0.003386
XGB F2 delta (GridSearchCV vs. notebook 12's RandomizedSearchCV): +0.111851

Which scaler setting won each search (expected to be close to a coin flip for both --
tree splits don't depend on feature scale):
  RF best scaler:  None
  XGB best scaler: StandardScaler()


## 6. Save the GridSearchCV models (new filenames -- notebook 12's saved models are untouched)

In [8]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(rf_best, "../models/random_forest_all_vehicles_gridsearch.joblib")
joblib.dump(xgb_best, "../models/xgboost_all_vehicles_gridsearch.joblib")
print("Saved ../models/random_forest_all_vehicles_gridsearch.joblib and "
      "../models/xgboost_all_vehicles_gridsearch.joblib")

Saved ../models/random_forest_all_vehicles_gridsearch.joblib and ../models/xgboost_all_vehicles_gridsearch.joblib


## 7. What this notebook establishes

A full, exhaustive, docs-informed hyperparameter search for both tree models, replacing notebook
12's sampled `RandomizedSearchCV` with `GridSearchCV` over the coach-specified grid (RF) and a
comparably-sized, XGBoost-docs-informed grid (XGBoost) -- per the coach's direct code review.
Results are quoted honestly against notebook 12's numbers once this notebook has actually been run;
notebook 12 and its saved models/results are untouched either way.